# 01 - Preprocess PhysioNet motor imagery

This notebook builds the run-aware dataset used in the utility and privacy experiments.

## Contents

1. Setup and paths
2. Subject processing function
3. Build the 1-50 subject dataset
4. Basic checks
5. Save and reload check

The saved file contains `X`, `y`, `subject_ids`, and `run_ids`.

## 1. Setup and paths

Mount Drive, install MNE, import packages, and set the dataset paths.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("Drive mount skipped. This is fine outside Colab.")

In [ ]:
!pip -q install mne

In [ ]:
from pathlib import Path
import gc

import mne
import numpy as np
from mne.datasets import eegbci

In [ ]:
DRIVE_ROOT = Path("/content/drive/MyDrive")
PROJECT_DIR = DRIVE_ROOT / "URV_Datasets" / "PhysioNet_MI"
PROCESSED_DIR = PROJECT_DIR / "processed"

DATA_ROOT = PROJECT_DIR
OUTPUT_FILE = DRIVE_ROOT / "URV_Datasets" / "physionet_mi_lr_imagery_subjects_1_50_with_runs.npz"

SUBJECTS = list(range(1, 51))
RUNS = [4, 8, 12]

TMIN = 0.0
TMAX = 4.0
BANDPASS = (8.0, 30.0)

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Output file:", OUTPUT_FILE)

## 2. Subject processing function

Each subject is processed run by run. Runs 4, 8, and 12 are left/right motor imagery runs in EEGBCI.
`T1` is mapped to `0`, and `T2` is mapped to `1`.

In [ ]:
def process_subject(subject_id, runs=RUNS):
    X_parts = []
    y_parts = []
    subject_parts = []
    run_parts = []

    for run in runs:
        try:
            file_path = eegbci.load_data(subject_id, [run], path=str(DATA_ROOT))[0]
            raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
            eegbci.standardize(raw)

            # Keep EEG channels only; auxiliary channels are not used here.
            raw.pick("eeg")
            raw.filter(BANDPASS[0], BANDPASS[1], fir_design="firwin", verbose=False)

            events, event_id = mne.events_from_annotations(raw, verbose=False)
            if "T1" not in event_id or "T2" not in event_id:
                print(f"Skipping subject {subject_id}, run {run}: missing T1/T2")
                continue

            epochs = mne.Epochs(
                raw,
                events,
                event_id={"T1": event_id["T1"], "T2": event_id["T2"]},
                tmin=TMIN,
                tmax=TMAX,
                baseline=None,
                preload=True,
                verbose=False,
            )

            # MNE includes both endpoints, so 0-4s at 160 Hz gives 641 samples.
            X_run = epochs.get_data().astype("float32")
            event_codes = epochs.events[:, -1]
            y_run = np.where(event_codes == event_id["T1"], 0, 1).astype("int64")

            X_parts.append(X_run)
            y_parts.append(y_run)
            subject_parts.append(np.full(len(y_run), subject_id, dtype="int64"))
            run_parts.append(np.full(len(y_run), run, dtype="int64"))

        except Exception as exc:
            print(f"Skipping subject {subject_id}, run {run}: {exc}")

    if not X_parts:
        return None

    return (
        np.concatenate(X_parts, axis=0),
        np.concatenate(y_parts, axis=0),
        np.concatenate(subject_parts, axis=0),
        np.concatenate(run_parts, axis=0),
    )

## 3. Build the dataset

Loop through subjects 1-50, collect the trials, and keep the subject and run label for every trial.

In [ ]:
X_list = []
y_list = []
subject_list = []
run_list = []
skipped_subjects = []

for subject_id in SUBJECTS:
    result = process_subject(subject_id)

    if result is None:
        skipped_subjects.append(subject_id)
        continue

    X_s, y_s, subject_s, run_s = result
    print(
        f"Subject {subject_id:02d}: X={X_s.shape}, "
        f"labels={dict(zip(*np.unique(y_s, return_counts=True)))}, "
        f"runs={dict(zip(*np.unique(run_s, return_counts=True)))}"
    )

    X_list.append(X_s)
    y_list.append(y_s)
    subject_list.append(subject_s)
    run_list.append(run_s)
    gc.collect()

if not X_list:
    raise RuntimeError("No subjects were processed.")

X = np.concatenate(X_list, axis=0).astype("float32")
y = np.concatenate(y_list, axis=0).astype("int64")
subject_ids = np.concatenate(subject_list, axis=0).astype("int64")
run_ids = np.concatenate(run_list, axis=0).astype("int64")

print("\nFinal shapes")
print("X:", X.shape)
print("y:", y.shape)
print("subject_ids:", subject_ids.shape)
print("run_ids:", run_ids.shape)
print("Skipped subjects:", skipped_subjects)

## 4. Basic checks

Check class balance, run counts, and whether subjects 1-50 are all present.

In [ ]:
def summarize_labels(name, values):
    unique, counts = np.unique(values, return_counts=True)
    print(name, dict(zip(unique.tolist(), counts.tolist())))


summarize_labels("Task labels", y)
summarize_labels("Runs", run_ids)

unique_subjects, subject_counts = np.unique(subject_ids, return_counts=True)
print("Number of subjects:", len(unique_subjects))
print("Trial count range:", int(subject_counts.min()), "to", int(subject_counts.max()))

expected_subjects = set(SUBJECTS)
actual_subjects = set(unique_subjects.tolist())
print("Missing subjects:", sorted(expected_subjects - actual_subjects))
print("Extra subjects:", sorted(actual_subjects - expected_subjects))

## 5. Save and reload check

Save the `.npz` file, then reopen it once to confirm the expected keys and shapes.

In [ ]:
np.savez_compressed(
    OUTPUT_FILE,
    X=X,
    y=y,
    subject_ids=subject_ids,
    run_ids=run_ids,
)

print("Saved:", OUTPUT_FILE)

In [ ]:
check = np.load(OUTPUT_FILE)
print("Keys:", check.files)
print("X:", check["X"].shape)
print("y:", check["y"].shape)
print("subject_ids:", check["subject_ids"].shape)
print("run_ids:", check["run_ids"].shape)
print("Runs:", dict(zip(*np.unique(check["run_ids"], return_counts=True))))